# Rappel de l objectif rendefinssant les euristic , les meta heur, diff entre diversification et intensification.

## Data set cf. GitHub : https://github.com/PyVRP/VRPLIB

In [69]:
import vrplib

# Remplacez par le vrai nom d'un de vos fichiers (ex: data/C101.txt)
instance = vrplib.read_instance("data/C101.txt", instance_format="solomon") 
solution_opt = vrplib.read_solution("data/C101.sol")

# Afficher ce que contient l'instance
print(instance.keys())
print("Score optimal à battre :", solution_opt.get("cost", solution_opt.get("Cost")))

dict_keys(['name', 'vehicles', 'capacity', 'node_coord', 'demand', 'time_window', 'service_time', 'edge_weight'])
Score optimal à battre : 827.3


## Construction du Graph 


In [70]:
import random
from collections import deque

# =======================================================
# 0. STRUCTURE DE DONNÉES (Classes)
# =======================================================

class Node:
    """Modélise un sommet du réseau (Dépôt ou Client) avec ses paramètres."""
    def __init__(self, node_id, demand=0, earliest_time=0, latest_time=float('inf'), service_time=0):
        self.node_id = node_id  # Indice i
        self.q = demand         # Demande q_i
        self.e = earliest_time  # Ouverture de la fenêtre e_i
        self.l = latest_time    # Fermeture de la fenêtre l_i
        self.s = service_time   # Temps de service s_i

class CVRPTW_Network:
    """Modélise le graphe orienté complet G = (V, E) et les paramètres du CVRPTW."""
    def __init__(self, n_customers, k_trucks, capacity):
        self.N = n_customers    # Nombre total de clients N
        self.K = k_trucks       # Nombre de camions K
        self.Q = capacity       # Capacité maximale d'un camion Q
        self.B = float('inf')   # Borne de coût B'

        self.V = list(range(self.N + 1))
        self.C = [node for node in self.V if node != 0]
        self.E = [(i, j) for i in self.V for j in self.V if i != j]

        self.nodes = {}
        self.travel_times = {}

    def add_node(self, node_id, demand=0, e=0, l=float('inf'), s=0):
        self.nodes[node_id] = Node(node_id, demand, e, l, s)

    def set_travel_time(self, i, j, time_val):
        if (i, j) in self.E:
            self.travel_times[(i, j)] = time_val

    def calculate_objective(self, routes):
        """Calcule la fonction objectif Z = Somme(C_ij * X_ij)."""
        total_time = 0
        for route in routes:
            for idx in range(len(route) - 1):
                i = route[idx]
                j = route[idx + 1]
                total_time += self.travel_times.get((i, j), 0)
        return total_time

## Verificaton des fenetres de temps

In [71]:
def est_realisable(route_tuple, reseau):
    """Vérifie si une route respecte les capacités et fenêtres de temps."""
    charge_actuelle = 0
    temps_actuel = 0
    
    for k in range(len(route_tuple) - 1):
        i = route_tuple[k]
        j = route_tuple[k+1]
        
        charge_actuelle += reseau.nodes[i].q
        if charge_actuelle > reseau.Q:
            return False
            
        temps_actuel = max(temps_actuel, reseau.nodes[i].e)
        temps_actuel += reseau.nodes[i].s + reseau.travel_times.get((i, j), float('inf'))
        
        if temps_actuel > reseau.nodes[j].l:
            return False 
            
    return True

## Choix de la méthode de Voisinage

In [72]:
# =======================================================
# LES 3 OPÉRATEURS DE RECHERCHE LOCALE (VOISINAGE)
# =======================================================

def voisinage_swap(route_tuple):
    """
    Méthode 1 : SWAP (Échange)
    Intervertit la position de deux clients sans toucher au reste.
    """
    route = list(route_tuple)
    n = len(route)
    
    # On parcourt les clients (on ignore le dépôt au début [0] et à la fin [n-1])
    for i in range(1, n - 2):
        for j in range(i + 1, n - 1):
            voisin = route[:]
            # Échange des valeurs aux index i et j
            voisin[i], voisin[j] = voisin[j], voisin[i]
            
            yield tuple(voisin)


def voisinage_relocate(route_tuple):
    """
    Méthode 2 : RELOCATE (Insertion)
    Retire un client de sa position actuelle pour l'insérer ailleurs.
    """
    route = list(route_tuple)
    n = len(route)
    
    for i in range(1, n - 1):
        for j in range(1, n - 1):
            if i != j:
                voisin = route[:]
                # 1. On retire le client à l'index i
                client_a_deplacer = voisin.pop(i)
                # 2. On l'insère à son nouvel index j
                voisin.insert(j, client_a_deplacer)
                
                yield tuple(voisin)


def voisinage_2opt(route_tuple):
    """
    Méthode 3 : 2-OPT (Inversion de segment)
    Inverse totalement l'ordre de passage d'un sous-groupe de clients[cite: 11].
    """
    route = list(route_tuple)
    n = len(route)
    
    for i in range(1, n - 2):
        for j in range(i + 1, n - 1):
            voisin = route[:]
            # On extrait le segment de i à j (inclus), on l'inverse, et on le remplace
            voisin[i:j+1] = reversed(voisin[i:j+1])
            
            yield tuple(voisin)

# =======================================================
# LE SÉLECTEUR DE MÉTHODE (À utiliser dans vos boucles)
# =======================================================

def voisinage(route_tuple, methode="swap"):
    """
    Fonction unifiée pour générer le voisinage selon la méthode choisie.
    methode acceptée : "swap", "relocate", "2opt"
    """
    if methode == "swap":
        return voisinage_swap(route_tuple)
    elif methode == "relocate":
        return voisinage_relocate(route_tuple)
    elif methode == "2opt":
        return voisinage_2opt(route_tuple)
    else:
        raise ValueError(f"Méthode '{methode}' inconnue. Choisissez 'swap', 'relocate' ou '2opt'.")

## Multi Start / GARSP ? 
Explication du principe et on l implémente pour chaque algo(c est une hypothèse)

In [73]:
def random_solution(reseau):
    """Génère une solution aléatoire valide (Multi-Start)."""
    clients = reseau.C[:]
    max_essais = 2000
    
    for _ in range(max_essais):
        random.shuffle(clients)
        candidat = tuple([0] + clients + [0])
        if est_realisable(candidat, reseau):
            return candidat
            
    raise Exception("Impossible de trouver une solution initiale réalisable.")

## Heurisitcs 

### Hills Climbings

### A*

In [74]:
def hill_climbing(solution_initiale, reseau,methode_voisinage="swap"):
    """Heuristique classique : Boucle jusqu'à l'optimum local."""
    solution_courante = solution_initiale
    nouveau = True
    
    while nouveau:
        meilleure_solution = solution_courante
        meilleur_score = reseau.calculate_objective([list(solution_courante)])
        
        for voisin in voisinage(solution_courante, methode=methode_voisinage):
            if est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective([list(voisin)])
                if score_voisin < meilleur_score:
                    meilleure_solution = voisin
                    meilleur_score = score_voisin
                    
        nouveau = (meilleure_solution != solution_courante)
        solution_courante = meilleure_solution
    return solution_courante

## Meta - Heurisitcs

### Tabu Search

In [75]:
def recherche_tabou(solution_initiale, reseau, taille_tabou, iter_max, methode_voisinage="swap"):
    """Métaheuristique Tabou avec liste mémoire pour éviter les blocages locaux."""
    liste_tabou = deque(maxlen=taille_tabou)
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    valeur_meilleure_globale = reseau.calculate_objective([list(solution_initiale)])
    nb_iter_sans_amelioration = 0
    
    while nb_iter_sans_amelioration < iter_max:
        meilleur_voisin = None
        valeur_meilleur_voisin = float('inf')
        
        for voisin in voisinage(solution_courante, methode=methode_voisinage):
            if voisin not in liste_tabou and est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective([list(voisin)])
                if score_voisin < valeur_meilleur_voisin:
                    meilleur_voisin = voisin
                    valeur_meilleur_voisin = score_voisin
        
        if meilleur_voisin is None:
            break
            
        if valeur_meilleur_voisin < valeur_meilleure_globale:
            meilleure_globale = meilleur_voisin
            valeur_meilleure_globale = valeur_meilleur_voisin
            nb_iter_sans_amelioration = 0
        else:
            nb_iter_sans_amelioration += 1
            
        solution_courante = meilleur_voisin
        liste_tabou.append(solution_courante)
        
    return meilleure_globale

### Simulated Annealing

In [76]:
import math
import random
import time

def recuit_simule(solution_initiale, reseau, temp_initiale, taux_refroidissement, iter_max, methode_voisinage="swap"):
    """
    Métaheuristique : Recuit Simulé (Simulated Annealing) pour le CVRPTW[cite: 11].
    Accepte des dégradations de score (détours) selon une probabilité qui baisse avec le temps.
    """
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # On calcule le coût initial avec la fonction objectif du réseau (MINIMISATION)[cite: 7]
    valeur_courante = reseau.calculate_objective([list(solution_courante)])
    valeur_meilleure_globale = valeur_courante
    
    temperature = temp_initiale
    
    for _ in range(iter_max):
        # On liste tous les voisins générés par la méthode choisie qui respectent Q et les fenêtres de temps[cite: 6, 7]
        voisins_valides = [v for v in voisinage(solution_courante, methode=methode_voisinage) if est_realisable(v, reseau)]
        
        if not voisins_valides:
            break # Sécurité : aucun voisin possible sans violer les contraintes
            
        # 1. On tire UN SEUL voisin au hasard
        voisin_choisi = random.choice(voisins_valides)
        valeur_voisin = reseau.calculate_objective([list(voisin_choisi)])
        
        # 2. Calcul de la différence de score
        # delta < 0 signifie que le voisin est plus rapide (amélioration)
        # delta > 0 signifie que le voisin est plus lent (dégradation)
        delta = valeur_voisin - valeur_courante
        
        # 3. Critère d'acceptation (Metropolis)
        if delta < 0:
            # C'est meilleur : on accepte d'office l'amélioration
            solution_courante = voisin_choisi
            valeur_courante = valeur_voisin
        else:
            # C'est pire : on calcule la probabilité d'accepter ce détour
            if temperature > 0.001:
                # Formule pour la minimisation : exp(-delta / T)
                probabilite = math.exp(-delta / temperature)
                
                # Le coup de dés !
                if random.random() < probabilite:
                    solution_courante = voisin_choisi
                    valeur_courante = valeur_voisin
                    
        # 4. Mise à jour du record absolu (on cherche le plus petit temps possible)
        if valeur_courante < valeur_meilleure_globale:
            meilleure_globale = solution_courante
            valeur_meilleure_globale = valeur_courante
            
        # 5. On refroidit la température
        temperature *= taux_refroidissement
        
    return meilleure_globale

## Test et print console

In [77]:
cvrptw_test = CVRPTW_Network(n_customers=4, k_trucks=1, capacity=50)

# Dépôt (0) et Clients (1 à 4)
cvrptw_test.add_node(0, demand=0, e=0, l=200, s=0)
cvrptw_test.add_node(1, demand=10, e=10, l=50, s=5)
cvrptw_test.add_node(2, demand=15, e=20, l=70, s=5)
cvrptw_test.add_node(3, demand=20, e=0, l=80, s=5)
cvrptw_test.add_node(4, demand=5, e=40, l=100, s=5)

# Génération de temps de trajets
random.seed(42)
for i in cvrptw_test.V:
    for j in cvrptw_test.V:
        if i != j:
            cvrptw_test.set_travel_time(i, j, random.randint(10, 30))

try:
    sol_init = random_solution(cvrptw_test)
    score_init = cvrptw_test.calculate_objective([list(sol_init)])
    print(f"Solution Initiale Aléatoire : {sol_init} | Temps : {score_init}")

    sol_hc = hill_climbing(sol_init, cvrptw_test)
    score_hc = cvrptw_test.calculate_objective([list(sol_hc)])
    print(f"Solution Hill Climbing      : {sol_hc} | Temps : {score_hc}")

    sol_tabou = recherche_tabou(sol_init, cvrptw_test, taille_tabou=5, iter_max=15)
    score_tabou = cvrptw_test.calculate_objective([list(sol_tabou)])
    print(f"Solution Recherche Tabou    : {sol_tabou} | Temps : {score_tabou}")

except Exception as e:
    print(f"Erreur : {e}")



Solution Initiale Aléatoire : (0, 1, 3, 2, 4, 0) | Temps : 96
Solution Hill Climbing      : (0, 3, 2, 1, 4, 0) | Temps : 64
Solution Recherche Tabou    : (0, 3, 2, 1, 4, 0) | Temps : 64
